# 01 Data Audit and Manifest Preparation

This notebook audits the downloaded public datasets and identifies whether each dataset contains images, masks, annotations, metadata, or only classification-style folders.

The goal is to determine which datasets are actually usable for optic disc/cup segmentation.

In [14]:
from pathlib import Path
import sys
import os
import pandas as pd

In [15]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

PROJECT_ROOT

PosixPath('/sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy')

In [16]:
DATASET_KEYS = [
    "glaucoma_fundus_imaging_bundle",
    "papila",
]

DATASET_KEYS

['glaucoma_fundus_imaging_bundle', 'papila']

In [17]:
from glaucoma_segmentation.data.discover_files import inventory_registry_datasets

inventory_df, summary_df, candidate_df = inventory_registry_datasets(
    dataset_keys=DATASET_KEYS,
    save_reports=True,
)

summary_df

,dataset_key,extension,file_class,file_count,total_size_bytes
2,glaucoma_fundus_imaging_bundle,.jpg,image,10051,5013137525
6,glaucoma_fundus_imaging_bundle,.png,possible_mask_image,8631,39319157
0,glaucoma_fundus_imaging_bundle,.bmp,possible_mask_image,1200,3886650400
3,glaucoma_fundus_imaging_bundle,.json,possible_annotation,1023,1977090
4,glaucoma_fundus_imaging_bundle,.mat,possible_annotation,650,16796084
1,glaucoma_fundus_imaging_bundle,.csv,possible_annotation,3,115247
8,glaucoma_fundus_imaging_bundle,.tif,possible_mask_image,2,50146
5,glaucoma_fundus_imaging_bundle,.pkl,other,1,814
7,glaucoma_fundus_imaging_bundle,.pth,other,1,1107541735
15,papila,.txt,possible_annotation,1963,1091265


In [18]:
candidate_df[
    [
        "dataset_key",
        "relative_path",
        "extension",
        "file_class",
        "has_mask_keyword",
        "file_size_bytes",
    ]
].head(100)

,dataset_key,relative_path,extension,file_class,has_mask_keyword,file_size_bytes
0,glaucoma_fundus_imaging_bundle,G1020/G1020.csv,.csv,possible_annotation,False,18036
2,glaucoma_fundus_imaging_bundle,G1020/Images/image_0.json,.json,possible_annotation,False,1083
4,glaucoma_fundus_imaging_bundle,G1020/Images/image_1.json,.json,possible_annotation,False,1157
6,glaucoma_fundus_imaging_bundle,G1020/Images/image_10.json,.json,possible_annotation,False,848
8,glaucoma_fundus_imaging_bundle,G1020/Images/image_1000.json,.json,possible_annotation,False,903
...,...,...,...,...,...,...
190,glaucoma_fundus_imaging_bundle,G1020/Images/image_1260.json,.json,possible_annotation,False,895
192,glaucoma_fundus_imaging_bundle,G1020/Images/image_1261.json,.json,possible_annotation,False,1416
194,glaucoma_fundus_imaging_bundle,G1020/Images/image_1270.json,.json,possible_annotation,False,1731
196,glaucoma_fundus_imaging_bundle,G1020/Images/image_1271.json,.json,possible_annotation,False,1062


In [19]:
summary_df.sort_values(
    ["dataset_key", "file_count"],
    ascending=[True, False],
)

,dataset_key,extension,file_class,file_count,total_size_bytes
2,glaucoma_fundus_imaging_bundle,.jpg,image,10051,5013137525
6,glaucoma_fundus_imaging_bundle,.png,possible_mask_image,8631,39319157
0,glaucoma_fundus_imaging_bundle,.bmp,possible_mask_image,1200,3886650400
3,glaucoma_fundus_imaging_bundle,.json,possible_annotation,1023,1977090
4,glaucoma_fundus_imaging_bundle,.mat,possible_annotation,650,16796084
1,glaucoma_fundus_imaging_bundle,.csv,possible_annotation,3,115247
8,glaucoma_fundus_imaging_bundle,.tif,possible_mask_image,2,50146
5,glaucoma_fundus_imaging_bundle,.pkl,other,1,814
7,glaucoma_fundus_imaging_bundle,.pth,other,1,1107541735
15,papila,.txt,possible_annotation,1963,1091265


In [20]:
dataset_summary = (
    inventory_df
    .groupby(["dataset_key", "file_class"])
    .size()
    .reset_index(name="file_count")
    .pivot(index="dataset_key", columns="file_class", values="file_count")
    .fillna(0)
    .astype(int)
)

dataset_summary

file_class,image,other,possible_annotation,possible_mask_image
dataset_key,,,,
glaucoma_fundus_imaging_bundle,10051,2,1676,9833
papila,488,4,1985,488


## Initial audit complete

The next step is to review the candidate masks/annotations and determine which datasets can support optic disc/cup segmentation.

After that, the manifest-building portion of this notebook will create structured image/mask pair records.

## Path-pattern inspection

Inspect folder patterns inside the audited datasets so we can identify likely fundus image folders, mask folders, and annotation folders before writing manifest-building code.

In [21]:
# Find project root from wherever the notebook is opened
def find_project_root(start_path: Path | None = None) -> Path:
    start_path = start_path or Path.cwd()
    for path in [start_path, *start_path.parents]:
        if (path / "configs").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root containing configs/ and src/")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from glaucoma_segmentation.data.discover_files import inventory_registry_datasets

DATASET_KEYS = ["glaucoma_fundus_imaging_bundle", "papila"]

inventory_df, summary_df, candidate_df = inventory_registry_datasets(
    dataset_keys=DATASET_KEYS,
    save_reports=True,
)

summary_df

,dataset_key,extension,file_class,file_count,total_size_bytes
2,glaucoma_fundus_imaging_bundle,.jpg,image,10051,5013137525
6,glaucoma_fundus_imaging_bundle,.png,possible_mask_image,8631,39319157
0,glaucoma_fundus_imaging_bundle,.bmp,possible_mask_image,1200,3886650400
3,glaucoma_fundus_imaging_bundle,.json,possible_annotation,1023,1977090
4,glaucoma_fundus_imaging_bundle,.mat,possible_annotation,650,16796084
1,glaucoma_fundus_imaging_bundle,.csv,possible_annotation,3,115247
8,glaucoma_fundus_imaging_bundle,.tif,possible_mask_image,2,50146
5,glaucoma_fundus_imaging_bundle,.pkl,other,1,814
7,glaucoma_fundus_imaging_bundle,.pth,other,1,1107541735
15,papila,.txt,possible_annotation,1963,1091265


In [22]:
path_parts = candidate_df["relative_path"].astype(str).str.split("/", expand=True)

path_patterns = (
    candidate_df
    .assign(
        top_folder=path_parts[0],
        second_folder=path_parts[1].fillna(""),
        third_folder=path_parts[2].fillna("")
    )
    .groupby(["dataset_key", "top_folder", "second_folder", "file_class"])
    .size()
    .reset_index(name="n_files")
    .sort_values(["dataset_key", "n_files"], ascending=[True, False])
)

output_path = PROJECT_ROOT / "reports" / "data_audit" / "public_dataset_path_patterns.csv"
path_patterns.to_csv(output_path, index=False)

print(f"Saved path-pattern summary to: {output_path}")
path_patterns.head(75)

Saved path-pattern summary to: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/reports/data_audit/public_dataset_path_patterns.csv


,dataset_key,top_folder,second_folder,file_class,n_files
12,glaucoma_fundus_imaging_bundle,REFUGE,Masks_Square,possible_mask_image,1200
14,glaucoma_fundus_imaging_bundle,REFUGE,test,possible_mask_image,1200
16,glaucoma_fundus_imaging_bundle,REFUGE,train,possible_mask_image,1200
18,glaucoma_fundus_imaging_bundle,REFUGE,val,possible_mask_image,1200
3,glaucoma_fundus_imaging_bundle,G1020,Masks_Cropped,possible_mask_image,1040
1,glaucoma_fundus_imaging_bundle,G1020,Images,possible_annotation,1020
2,glaucoma_fundus_imaging_bundle,G1020,Masks,possible_mask_image,1020
4,glaucoma_fundus_imaging_bundle,G1020,Masks_Square,possible_mask_image,1020
7,glaucoma_fundus_imaging_bundle,ORIGA,Masks_Square,possible_mask_image,651
5,glaucoma_fundus_imaging_bundle,ORIGA,Masks,possible_mask_image,650


In [23]:
image_like_patterns = (
    path_patterns[
        path_patterns["file_class"].isin(
            ["image", "possible_mask_image", "possible_annotation"]
        )
    ]
    .sort_values(
        ["dataset_key", "top_folder", "second_folder", "file_class"]
    )
)

output_path = PROJECT_ROOT / "reports" / "data_audit" / "public_dataset_image_like_path_patterns.csv"
image_like_patterns.to_csv(output_path, index=False)

print(f"Saved image-like path-pattern summary to: {output_path}")
image_like_patterns

Saved image-like path-pattern summary to: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/reports/data_audit/public_dataset_image_like_path_patterns.csv


,dataset_key,top_folder,second_folder,file_class,n_files
0,glaucoma_fundus_imaging_bundle,G1020,G1020.csv,possible_annotation,1
1,glaucoma_fundus_imaging_bundle,G1020,Images,possible_annotation,1020
2,glaucoma_fundus_imaging_bundle,G1020,Masks,possible_mask_image,1020
3,glaucoma_fundus_imaging_bundle,G1020,Masks_Cropped,possible_mask_image,1040
4,glaucoma_fundus_imaging_bundle,G1020,Masks_Square,possible_mask_image,1020
5,glaucoma_fundus_imaging_bundle,ORIGA,Masks,possible_mask_image,650
6,glaucoma_fundus_imaging_bundle,ORIGA,Masks_Cropped,possible_mask_image,650
7,glaucoma_fundus_imaging_bundle,ORIGA,Masks_Square,possible_mask_image,651
8,glaucoma_fundus_imaging_bundle,ORIGA,OrigaList.csv,possible_annotation,1
9,glaucoma_fundus_imaging_bundle,ORIGA,Semi-automatic-annotations,possible_annotation,650
